# Two Screens, Different Statuses

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_OER/blob/main/course_materials/notebooks/08_polyglot_incident.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_OER/blob/main/course_materials/notebooks/08_polyglot_incident.ipynb)

Click **Open in Colab** to open this notebook directly. Save your working copy
in Drive. You can also use local Jupyter.

A staff dashboard says a ticket is resolved while the resident page says it is
open. We will follow the update, test a safe retry, and check the derived copy.

The notebook uses Python's built-in SQLite database and dictionaries. It makes
no network requests and needs no password or package installation. SQLite runs
a real local transaction; the dictionary experiments model a MongoDB projection
but do not connect to MongoDB or implement a concurrent consumer.

Run the cells in order. Section 1 supports the instructor demonstration. Section
2 contains the individual lab experiment. Section 3 broadens the investigation.
The weekly lab specifies **one Brightspace text response**, not a notebook upload.

## 1. One Transaction Records the Change and Its Event

An **authoritative record** establishes the accepted business state. An **outbox**
is a table of outgoing events written in the same transaction as that state.
The relay can deliver the recorded events later.

The initialization cell creates three synthetic tickets and an empty outbox.
Run it again only to restart the demonstration. It resets this in-memory database,
not any cloud project. Primary keys give each ticket and event a stable identity.
The unique ticket/version pair rejects two event IDs for the same accepted version.

In [ ]:
import sqlite3

if "source_db" in globals():
    source_db.close()
source_db = sqlite3.connect(":memory:")
source_db.row_factory = sqlite3.Row
source_db.execute("PRAGMA foreign_keys = ON")
source_db.executescript("""
CREATE TABLE tickets (
    ticket_id INTEGER PRIMARY KEY,
    status TEXT NOT NULL CHECK (status IN ('open', 'resolved', 'closed')),
    source_version INTEGER NOT NULL CHECK (source_version > 0)
);
CREATE TABLE outbox (
    event_id TEXT PRIMARY KEY,
    ticket_id INTEGER NOT NULL REFERENCES tickets(ticket_id),
    status TEXT NOT NULL,
    source_version INTEGER NOT NULL,
    UNIQUE (ticket_id, source_version)
);
""")
with source_db:
    source_db.executemany("INSERT INTO tickets VALUES (?, ?, ?)", [
        (1008, "open", 16), (1009, "resolved", 5), (1010, "resolved", 7)
    ])
print([dict(row) for row in source_db.execute("SELECT * FROM tickets ORDER BY ticket_id")])

### A Failure Before the Event Is Recorded

The update below runs inside a transaction. We deliberately raise an exception
before inserting its event. The `with source_db` block rolls back because the
exception leaves that block; the `except` then describes the simulated failure.

Predict the stored status and event count after rollback. Both the ticket update
and the event must belong to the same database transaction for this guarantee.
An HTTP call to another service would not join it automatically.

In [ ]:
try:
    with source_db:
        source_db.execute(
            "UPDATE tickets SET status = ?, source_version = ? WHERE ticket_id = ?",
            ("resolved", 17, 1008),
        )
        raise RuntimeError("Simulated application failure before the outbox insert")
except RuntimeError as error:
    print(error)

print("After rollback:", dict(source_db.execute(
    "SELECT * FROM tickets WHERE ticket_id = ?", (1008,)
).fetchone()))
print("Outbox rows:", source_db.execute("SELECT COUNT(*) FROM outbox").fetchone()[0])
# From fresh initialization: open, version 16, and zero events.

### A Successful Local Commit

This attempt writes the ticket and event inside one transaction. The version
condition applies this transition only from version 16. `rowcount` tells us whether
that row matched; rerunning the cell after success does not create another event.

The `?` placeholders carry values separately from SQL. This is SQLite parameter
syntax; Psycopg uses `%s`. The transaction/outbox idea transfers to PostgreSQL,
but this cell is an executed SQLite example, not a PostgreSQL connection.

In [ ]:
with source_db:
    changed = source_db.execute(
        "UPDATE tickets SET status = ?, source_version = ? "
        "WHERE ticket_id = ? AND source_version = ?",
        ("resolved", 17, 1008, 16),
    )
    if changed.rowcount == 1:
        source_db.execute("INSERT INTO outbox VALUES (?, ?, ?, ?)",
                          ("evt-1008-17", 1008, "resolved", 17))
    else:
        print("Version 16 no longer matches. Inspect the existing state.")

print("Authoritative:", dict(source_db.execute(
    "SELECT * FROM tickets WHERE ticket_id = ?", (1008,)
).fetchone()))
print("Recorded events:", [dict(row) for row in source_db.execute("SELECT * FROM outbox")])

### A Later Accepted Update

The paper incident in the lab stops at authoritative version 17. For the following
ordering experiment, suppose staff later close the ticket at version 18. This
next transaction records that later state and its event. It does not claim that
version 18 already existed at the paper incident's earlier observation time.

The outbox now has two outgoing events. **Recorded** does not mean delivered or
applied. We have not implemented a relay, queue, or acknowledgment protocol.

In [ ]:
with source_db:
    changed = source_db.execute(
        "UPDATE tickets SET status = ?, source_version = ? "
        "WHERE ticket_id = ? AND source_version = ?",
        ("closed", 18, 1008, 17),
    )
    if changed.rowcount == 1:
        source_db.execute("INSERT INTO outbox VALUES (?, ?, ?, ?)",
                          ("evt-1008-18", 1008, "closed", 18))
print("Authoritative:", dict(source_db.execute(
    "SELECT * FROM tickets WHERE ticket_id = ?", (1008,)
).fetchone()))
print("Recorded events:", source_db.execute("SELECT COUNT(*) FROM outbox").fetchone()[0])

## 2. Duplicate and Delayed Events

A **projection** is a derived record arranged for a particular read. Here the
resident's projection begins at version 16. Each event supplies the complete
projected status for ticket 1008. Versions come from the authoritative order for
that ticket, not the consumer's clock.

Read the delivery list before running it. The first version 17 should apply,
its duplicate should do nothing, old version 16 should do nothing, and version
18 should apply. The cell resets the projection at its start, so each experiment
uses the same starting state.

In [ ]:
ENFORCE_VERSION = True
projection = {"ticket_id": 1008, "status": "open", "source_version": 16}
deliveries = [
    {"event_id": "evt-1008-17", "status": "resolved", "source_version": 17},
    {"event_id": "evt-1008-17", "status": "resolved", "source_version": 17},
    {"event_id": "evt-1008-16", "status": "open", "source_version": 16},
    {"event_id": "evt-1008-18", "status": "closed", "source_version": 18},
]

for event in deliveries:
    if ENFORCE_VERSION and event["source_version"] <= projection["source_version"]:
        print("Ignore old or duplicate event:", event["event_id"])
    else:
        projection["status"] = event["status"]
        projection["source_version"] = event["source_version"]
        projection["last_event_id"] = event["event_id"]
        print("Apply:", event["event_id"])
    print("Resident now sees:", projection["status"], projection["source_version"])

### Your Experiment

Move the version 16 event to the **end** of `deliveries` and run that cell again.
Then set `ENFORCE_VERSION = False` and rerun it. Finally restore `True` and rerun.
Compare the final status and version in the broken and repaired model.

Keep the observed results for the lab's Brightspace incident update. You do not
need a separate notebook submission. A hypothetical colleague is the audience
for the writing; the work is individual.

The guard assumes complete-state events and one ordered version sequence per
ticket. An event that says "add two" is different from one that says "the total
is two." Skipping an earlier increment can lose an effect. A production consumer
also needs an atomic comparison-and-update; this Python read/write loop models
the decision with one consumer only.

## 3. The Other Records Matter Too

After repairing ticket 1008, inspect a wider scope. The following three-document
projection contains a missing ticket, an unexpected ticket, and a wrong status
whose version number happens to match. Its count equals the authoritative count.

The query reads all three authoritative tickets from SQLite. We compare only
the owned fields `status` and `source_version`, keyed by `ticket_id`. Projection-only
metadata such as `last_event_id` has a different purpose and is not compared as
though it were an authoritative business field.

In [ ]:
source_records = {
    row["ticket_id"]: {"status": row["status"], "source_version": row["source_version"]}
    for row in source_db.execute("SELECT * FROM tickets ORDER BY ticket_id")
}
resident_records = {
    1008: {"status": projection["status"], "source_version": projection["source_version"]},
    1010: {"status": "open", "source_version": 7},
    9999: {"status": "open", "source_version": 1},
}
print("Counts:", len(source_records), len(resident_records))
findings = []
for ticket_id in sorted(source_records.keys() | resident_records.keys()):
    expected = source_records.get(ticket_id)
    observed = resident_records.get(ticket_id)
    if expected is None:
        finding = "unexpected projection ID"
    elif observed is None:
        finding = "missing projection"
    elif observed != expected:
        finding = f"mismatch: expected {expected}, found {observed}"
    else:
        continue
    findings.append((ticket_id, finding))
    print(ticket_id, finding)
print("Records needing investigation:", len(findings))

### A Rebuild Has an Explicit Direction

In this small model, all compared fields belong to `source_records`, so we can
rebuild a separate candidate from that source and compare its IDs and values.
The source stays unchanged. After restoring the version guard, expect three
findings before the rebuild and zero afterward. If the guard remains broken,
ticket 1008 adds another mismatch.

This dictionary assignment is not a production cutover procedure. Before
replacing a live projection, preserve independently owned fields, account for
new source writes during the rebuild, verify the candidate, and switch readers
through a controlled operation. A lagging source read must not overwrite newer
accepted state. Equal versions with different values deserve investigation.

In [ ]:
rebuilt_records = {ticket_id: dict(fields) for ticket_id, fields in source_records.items()}
print("Same IDs:", rebuilt_records.keys() == source_records.keys())
print("Same owned field values:", rebuilt_records == source_records)
remaining = [ticket_id for ticket_id in source_records.keys() | rebuilt_records.keys()
             if source_records.get(ticket_id) != rebuilt_records.get(ticket_id)]
print("Remaining mismatches:", remaining)

## The Corresponding MongoDB Operation

For an **existing** document with `_id: 1008`, MongoDB can compare the stored
version and update that document atomically. This reference fragment is not
executed by this notebook:

```javascript
db.ticket_projection.updateOne(
  { _id: 1008, source_version: { $lt: 18 } },
  { $set: { status: "closed", source_version: 18,
            last_event_id: "evt-1008-18" } }
)
```

The filter and update belong to one command. An older or duplicate delivery
cannot lower the stored version. `matchedCount: 0` still needs interpretation:
the document may already be at version 18 or later, or it may be absent. A
missing document needs an explicit initialization policy. Do not blindly add
`upsert: true` to this version-filtered command: an existing `_id` with a newer
version would fail the filter and collide with a proposed inserted `_id`.

An equal-version record with the wrong status also fails the filter. Reconciliation
must identify and repair that corruption deliberately; retries alone will not.

## Project Clinic Connection

Use the canonical final-project assignment rather than adding a second project.
If your project uses one database, identify an important query result and one
operation whose failure would matter. If it uses two, name the owner of each
duplicated fact and the repair direction.

The same investigation connects to access and recovery: restore the consumer's
approved, narrowly scoped access; replay recorded work safely; verify the
affected IDs and fields; and inspect the wider backlog. A public browser must
not receive a privileged database credential. A projection that can be rebuilt
from its authoritative source has a different recovery plan from an independent
historical record.

## Optional Extension: Deltas and Missing Versions

Suppose version 17 means "add one item" and version 18 means "add two items."
If 18 arrives first and a version guard then ignores 17, what effect is lost?
Describe how complete-state events, ordered replay, or reconstruction could
change the design. This is optional practice, not another submission.

## Cleanup

Close the local in-memory database when finished. This notebook opened no cloud
resources, IP rules, or credentials. To repeat the demonstration, start again
with the initialization cell in Section 1.

In [ ]:
source_db.close()
print("Closed the notebook's local database. No cloud resources were created.")

## Sources and Reuse

- [AWS transactional outbox explanation](https://docs.aws.amazon.com/prescriptive-guidance/latest/cloud-design-patterns/transactional-outbox.html)
- [PostgreSQL transaction tutorial](https://www.postgresql.org/docs/current/tutorial-transactions.html)
- [MongoDB single-document atomicity](https://www.mongodb.com/docs/manual/core/write-operations-atomicity/)
- [Python sqlite3 transaction control](https://docs.python.org/3/library/sqlite3.html#transaction-control)

The synthetic example illustrates these mechanisms without implementing a
distributed messaging platform. Course prose: CC BY-NC-SA 4.0. Code: MIT.
Synthetic records: CC0.